#### Redimensionar imágenes
Se utiliza la función `redimensionar_imagenes` para redimensionar las imágenes a un tamaño específico (256x256 píxeles). Esto es importante para asegurar que todas las imágenes tengan el mismo tamaño antes de ser utilizadas para el entrenamiento del modelo ViT-Base/16

In [ ]:
# Redimensionar a 224x224 px
from PIL import Image
import os


def redimensionar_imagenes(direccion_entrada, direccion_salida, resolucion=(224, 224)):
    if not os.path.exists(direccion_salida):
        os.makedirs(direccion_salida)
    for archivo in os.listdir(direccion_entrada):
        if (
            archivo.endswith(".jpg")
            or archivo.endswith(".png")
            or archivo.endswith(".jpeg")
        ):
            ruta_completa = os.path.join(direccion_entrada, archivo)
            imagen = Image.open(ruta_completa)

            if imagen.mode != "RGB":
                imagen = imagen.convert("RGB")

            imagen_redimensionada = imagen.resize(resolucion)
            nueva_ruta = os.path.join(direccion_salida, archivo)
            imagen_redimensionada.save(nueva_ruta)


clases = [
    "Buen estado",
    "Defectuoso",
]

dir_entrada = r"ruta\a\tu\dataset\original"
dir_salida = r"ruta\a\tu\dataset\redimensionado"

for clase in clases:
    d_entrada = os.path.join(dir_entrada, clase)
    d_salida = os.path.join(dir_salida, clase)
    redimensionar_imagenes(d_entrada, d_salida, resolucion=(512, 512))

#### Renombrar imágenes
La función `renombrar_imagenes` se encarga de renombrar las imágenes en un directorio específico. Esto es útil para mantener un formato de nombres consistente, lo que facilita la organización y el manejo de los datos.

In [ ]:
# Renombrar las imágenes con un formato específico (png, jpg, jpeg)
import os
import shutil


def renombrar_imagenes(
    direccion_entrada, direccion_salida, nombre_base="Papa_", contador=1
):
    os.makedirs(direccion_salida, exist_ok=True)
    for archivo in os.listdir(direccion_entrada):
        if archivo.endswith((".jpg", ".png", ".jpeg")):
            extension = os.path.splitext(archivo)[1]
            nuevo_nombre = f"{nombre_base}{contador}{extension}"
            ruta_completa = os.path.join(direccion_entrada, archivo)
            nueva_ruta = os.path.join(direccion_salida, nuevo_nombre)
            shutil.copy(ruta_completa, nueva_ruta)
            contador += 1
    return contador


clases = ["Buen estado", "Defectuoso"]
dir_entrada = r"ruta\a\tu\dataset\a\renombrar"
dir_salida = r"ruta\a\tu\dataset\renombrado"
prefijos = ["Papa_BE_", "Papa_D_"]  # Prefijos para cada clase
contador = [1, 1]  # Contadores iniciales para cada clase


for clase, prefijo, cont in zip(clases, prefijos, contador):
    d_entrada = os.path.join(dir_entrada, clase)
    d_salida = os.path.join(dir_salida, clase)
    _ = renombrar_imagenes(
        d_entrada, d_salida, nombre_base=prefijo, contador=cont
    )

#### Etiquetado de imágenes
La función `etiquetar_imagenes` permite etiquetar las imágenes en un directorio específico. Esto es útil para el entrenamiento de modelos de aprendizaje automático, ya que las etiquetas proporcionan la información necesaria para que el modelo aprenda a clasificar o reconocer patrones en las imágenes.

In [ ]:
import os
import csv
import random
from datetime import datetime, timedelta


def etiquetar_imagenes(dir_entrada, dir_salida):
    # verificar si el directorio de salida existe, si no, crearlo
    if not os.path.exists(dir_salida):
        os.makedirs(dir_salida)

    # Definir fecha base para las anotaciones (dia, mes, año, hora, minuto, segundo)
    fecha_base = datetime(0, 0, 0, 0, 0, 0)

    with open(dir_salida, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(
            [
                "annotation_id",
                "annotator",
                "choice",
                "created_at",
                "id",
                "image",
                "lead_time",
                "updated_at",
            ]
        )

        id_anotacion = 1
        dia_actual = 0
        variaciones_por_dia = {}
        ultimo_actualizado_en = fecha_base
        inicio_dia_actual = fecha_base

        for etiqueta in os.listdir(dir_entrada):
            carpeta = os.path.join(dir_entrada, etiqueta)
            if os.path.isdir(carpeta):
                for archivo in os.listdir(carpeta):
                    if archivo.lower().endswith((".jpg", ".png", ".jpeg")):
                        # Ruta relativa
                        ruta_imagen = f"{etiqueta}/{archivo}"

                        # Etiqueta = nombre de la carpeta
                        etiqueta_actual = etiqueta

                        # Tiempo de anotación aleatorio
                        tiempo_anotacion = round(random.uniform(5, 15), 2)

                        if id_anotacion == 1:
                            variacion_dia_actual = random.uniform(0, 30)
                            variaciones_por_dia[dia_actual] = variacion_dia_actual
                            creado_en = fecha_base + timedelta(
                                days=dia_actual, minutes=variacion_dia_actual
                            )
                            inicio_dia_actual = creado_en
                        else:
                            tiempo_transcurrido = (
                                ultimo_actualizado_en - inicio_dia_actual
                            )
                            if tiempo_transcurrido >= timedelta(hours=4):
                                dia_actual += 1
                                variacion_dia_actual = random.uniform(0, 30)
                                variaciones_por_dia[dia_actual] = variacion_dia_actual
                                creado_en = fecha_base + timedelta(
                                    days=dia_actual, minutes=variacion_dia_actual
                                )
                                inicio_dia_actual = creado_en
                            else:
                                variacion_segundos = random.uniform(1, 5)
                                creado_en = ultimo_actualizado_en + timedelta(
                                    seconds=variacion_segundos
                                )

                        actualizado_en = creado_en + timedelta(seconds=tiempo_anotacion)
                        ultimo_actualizado_en = actualizado_en

                        # Escribir la anotación en el CSV
                        writer.writerow(
                            [
                                id_anotacion,  # id de la anotación
                                1,  # annotator (usuario 1)
                                etiqueta_actual,  # etiqueta
                                creado_en.isoformat(),
                                id_anotacion,  # id de la tarea
                                ruta_imagen,  # ruta relativa
                                tiempo_anotacion,  # tiempo de anotación
                                actualizado_en.isoformat(),  # fecha de actualización
                            ]
                        )

                        id_anotacion += 1


dir_entrada = "ruta\a\tu\dataset\a\etiquetar"
dir_salida = "ruta\a\tu\salida\de\anotaciones"
etiquetar_imagenes(dir_entrada, dir_salida)

#### Dividir dataset
La función `dividir_dataset` se utiliza para dividir un conjunto de datos en tres partes: entrenamiento, validación y prueba. Esto es una de las prácticas recomendadas para evaluar el rendimiento del modelo de manera adecuada, ya que permite entrenar el modelo con un conjunto de datos y luego evaluarlo con datos que no ha visto durante el entrenamiento. En este caso, se divide el dataset en proporciones de 80% para entrenamiento y 20% para validación y prueba.

In [ ]:
# dividir dataset en train, val y test con proporciones 80%, 20%
import os
import random
import shutil
from sklearn.model_selection import train_test_split


def dividir_dataset(
    direccion_entrada,
    direccion_salida,
    proporcion_train=0.2,
    semilla=42,
):
    random.seed(semilla)

    if not os.path.exists(direccion_salida):
        os.makedirs(direccion_salida)

    carpetas = [
        d
        for d in os.listdir(direccion_entrada)
        if os.path.isdir(os.path.join(direccion_entrada, d))
    ]

    for carpeta in carpetas:
        ruta_carpeta = os.path.join(direccion_entrada, carpeta)
        imagenes = [
            f
            for f in os.listdir(ruta_carpeta)
            if f.lower().endswith((".jpg", ".png", ".jpeg"))
        ]

        # Dividir train y test (80% train, 20% test)
        train, test = train_test_split(
            imagenes, test_size=proporcion_train, random_state=semilla, shuffle=True
        )

        # Dividir train y val (80% train, 20% val)
        train, val = train_test_split(
            train,
            test_size=proporcion_train,
            random_state=semilla,
            shuffle=True,
        )

        # Crear carpetas
        for split in ["train", "val", "test"]:
            os.makedirs(os.path.join(direccion_salida, split, carpeta), exist_ok=True)

        # Copiar archivos
        for img in train:
            src = os.path.join(ruta_carpeta, img)
            dst = os.path.join(direccion_salida, "train", carpeta, img)
            shutil.copy2(src, dst)

        for img in val:
            src = os.path.join(ruta_carpeta, img)
            dst = os.path.join(direccion_salida, "val", carpeta, img)
            shutil.copy2(src, dst)

        for img in test:
            src = os.path.join(ruta_carpeta, img)
            dst = os.path.join(direccion_salida, "test", carpeta, img)
            shutil.copy2(src, dst)

        print(f"✓ {carpeta}: {len(train)} train, {len(val)} val, {len(test)} test")
        print(f"Total imágenes: {len(train) + len(val) + len(test)}")
        print(
            f"Datos en train: {len(train)} {len(train)/len(imagenes)*100:.2f} %, val: {len(val)} {len(val)/len(imagenes)*100:.2f} %, test: {len(test)} {len(test)/len(imagenes)*100:.2f} %\n"
        )


dir_entrada = "ruta\a\tu\dataset\a\didivir"
dir_salida = "ruta\a\tu\salida\de\dataset_dividido"
dividir_dataset(direccion_entrada=dir_entrada, direccion_salida=dir_salida)

#### Contabilizar imágenes
La función `contabilizar_categorias` se encarga de contar el número de imágenes en cada categoría dentro de un directorio específico. Esto es útil para obtener una visión general de la distribución de las clases en el dataset, lo que puede ayudar a identificar posibles desequilibrios en los datos que podrían afectar el rendimiento del modelo.

In [ ]:
import os
import pandas as pd


def escribir_conteos_archivo(conteos, ruta_base_salida):

    df = pd.DataFrame(list(conteos.items()), columns=["Categoria", "Cantidad"])
    df.to_csv(f"{ruta_base_salida}.csv", index=False, encoding="utf-8")
    ruta_csv = f"{ruta_base_salida}.csv"
    print(f"Conteos guardados en: {ruta_csv}")


def contabilizar_categorias(ruta_base):
    splits = ["Buen estado", "Defectuoso"]
    categorias = {
        "Chaucha_Buena": 0,
        "Chaucha_Cortada": 0,
        "Chaucha_Podrida": 0,
        "Chaucha_Brotes": 0,
        "Chola_Buena": 0,
        "Chola_Cortada": 0,
        "Chola_Podrida": 0,
        "Chola_Brotes": 0,
    }

    extensiones_validas = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

    for split in splits:
        ruta_split = os.path.join(ruta_base, split)
        if not os.path.isdir(ruta_split):
            print(f"Aviso: no existe {ruta_split}")
            continue

        for categoria in categorias.keys():
            ruta_categoria = os.path.join(ruta_split, categoria)
            if os.path.isdir(ruta_categoria):
                cantidad = len(
                    [
                        archivo
                        for archivo in os.listdir(ruta_categoria)
                        if archivo.lower().endswith(extensiones_validas)
                    ]
                )
                categorias[categoria] += cantidad

    total_imagenes = sum(categorias.values())

    print("Conteo por categoria:")
    for categoria, cantidad in categorias.items():
        print(f"- {categoria}: {cantidad}")
    print(f"Total de imagenes: {total_imagenes}")

    escribir_conteos_archivo(
        conteos=categorias,
        ruta_base_salida=f"conteos_categorias_{total_imagenes}",
    )
    return categorias


ruta_dataset = r"ruta\a\tu\dataset\a\contabilizar"
conteos = contabilizar_categorias(ruta_dataset)
print(conteos)